# 03 · Modelos

Entrena el **banco temporal**: para evaluar cada mes de 201906 a 201910, entrena con los meses de la ventana de la configuración (en el `main`, todos los meses hasta dos antes) y guarda una predicción por mes y por semilla en `exp/pipeline/<corrida>/pred/`.

- En el `main` entrena el `main`, `AC` y A-consigna. **En un branch, entrena el branch y, si falta en esta máquina, el `main`.**
- Cada corrida guarda su configuración en `exp/pipeline/<corrida>/config.json`. **Si lo guardado es de otra configuración, corta** en lugar de reusarlo (por ejemplo, el `main` después de un merge).
- Una predicción que ya existe no se rehace: **si la VM se apaga, se vuelve a correr y sigue**.
- Los modelos que comparten la ventana se entrenan una vez (A-consigna: un modelo por semilla para los 3 meses).
- **Controles:** si la estrategia es `PAQ_P3`, `AC` o A-consigna y están `exp/22/` y `exp/15/`, las predicciones tienen que ser iguales bit a bit a las del `22` y el `15`. Y con la misma configuración, las huellas tienen que ser las versionadas en `resultados/pipeline/<corrida>_huellas_pred.csv` (reproducción entre máquinas).

In [1]:
import importlib
import time

import numpy as np
import pandas as pd

import warnings

import parametros as P
import funciones as F

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')   # falso positivo de pandas 2.2 con Python 3.14

importlib.reload(P)
importlib.reload(F)
P.RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Corrida:', P.CORRIDA['nombre'], '·', len(P.CORRIDA['bloques']), 'bloques')

Corrida: rango · 9 bloques


In [2]:
import json
import warnings

warnings.filterwarnings('ignore', message='Cannot add features from NoneType')
hiper = json.loads((P.RESULTADOS_ANTERIORES / '03_hiperparametros.json').read_text())
assert hiper['params'] == P.HIPER_03 and hiper['arboles'] == P.ARBOLES_03
info = F.leer_filas(P)
SEMILLAS = F.semillas(P, P.N_SEMILLAS)
configs = F.configuraciones(P)
firmas = {c['nombre']: F.revisar_config(P, c) for c in configs}   # corta si lo guardado es de otra configuración
print('Semillas:', SEMILLAS)
print('Configuraciones y firmas:', firmas)

Semillas: [294103, 342929, 161093, 677561, 431077]
Configuraciones y firmas: {'rango': 'bd0d50f9b13f68f8', 'main': '786f0055d685f102'}


## 1. Entrenamiento

In [3]:
segundos = {}
for c in configs:
    tabla = F.Tabla(P, c['bloques'])
    por_ventana = {}
    for m in c['meses']:
        por_ventana.setdefault(tuple(F.ventana(c, m, P)), []).append(m)
    for meses_train, meses_eval in por_ventana.items():
        falta = [(m, s) for m in meses_eval for s in SEMILLAS if not F.ruta_pred(P, c['nombre'], m, s).exists()]
        if not falta:
            continue
        entrena = np.isin(info['mes_fila'], meses_train)
        dset = F.dataset(tabla, entrena, info['clase'][entrena], c['peso_b2'], P)
        X_eval = {m: tabla.filas(info['mes_fila'] == m) for m in meses_eval}
        for s in SEMILLAS:
            if all(F.ruta_pred(P, c['nombre'], m, s).exists() for m in meses_eval):
                continue
            t0 = time.time()
            modelo = F.entrenar(dset, c, s, P)
            for m in meses_eval:
                F.guardar_npy(F.ruta_pred(P, c['nombre'], m, s),
                              modelo.predict(X_eval[m], num_threads=P.N_HILOS).astype(np.float32))
            segundos.setdefault(c['nombre'], []).append(time.time() - t0)
        print(f'{c["nombre"]} · entrena {meses_train[0]}-{meses_train[-1] % 100:02d} · evalúa {meses_eval} · '
              f'{len(tabla.columnas):,} columnas · {np.mean(segundos[c["nombre"]]):.0f} s por modelo · '
              f'memoria máxima {F.memoria_gb()} GB', flush=True)
        del dset, X_eval
print('Listo. Segundos por modelo:', {k: round(float(np.mean(v)), 1) for k, v in segundos.items()})

rango · entrena 201903-04 · evalúa [201906] · 1,782 columnas · 32 s por modelo · memoria máxima 5.8 GB
rango · entrena 201903-05 · evalúa [201907] · 1,782 columnas · 40 s por modelo · memoria máxima 6.5 GB
rango · entrena 201903-06 · evalúa [201908] · 1,782 columnas · 50 s por modelo · memoria máxima 6.5 GB
rango · entrena 201903-07 · evalúa [201909] · 1,782 columnas · 59 s por modelo · memoria máxima 6.5 GB
rango · entrena 201903-08 · evalúa [201910] · 1,782 columnas · 69 s por modelo · memoria máxima 6.5 GB
Listo. Segundos por modelo: {'rango': 68.5}


## 2. Huellas: reproducción del `22` y de lo versionado

In [4]:
for c in configs:
    pares = [(F.ruta_pred(P, c['nombre'], m, s), F.prediccion_anterior(P, c, m, s)) for m in c['meses'] for s in SEMILLAS]
    pares = [(a, b) for a, b in pares if b is not None and b.exists()]
    if pares:
        iguales = sum(np.array_equal(np.load(a), np.load(b)) for a, b in pares)
        assert iguales == len(pares), f'{c["nombre"]}: {iguales} de {len(pares)} predicciones iguales a las del 22 y el 15'
        print(f'{c["nombre"]}: {iguales} de {len(pares)} predicciones iguales bit a bit a las del 22 y el 15')
    reemplazar = P.CORRIDA['nombre'] == 'main' or c['nombre'] == P.CORRIDA['nombre']
    print(f'{c["nombre"]}: huellas {F.revisar_huellas_pred(P, c, firmas[c["nombre"]], SEMILLAS, reemplazar)}')

rango: huellas guardadas por primera vez
main: 25 de 25 predicciones iguales bit a bit a las del 22 y el 15
main: huellas 25 iguales bit a bit a las versionadas
